In [ ]:
pip install ir_datasets --user

Note: you may need to restart the kernel to use updated packages.


In [2]:
!pip install ir_datasets

In [3]:
import ir_datasets

cranfield = ir_datasets.load("cranfield")
vaswani = ir_datasets.load("vaswani")

print(cranfield)
print(vaswani)

Dataset(id='cranfield', provides=['docs', 'queries', 'qrels'])
Dataset(id='vaswani', provides=['docs', 'queries', 'qrels'])


In [4]:
print("Cranfield docs:", cranfield.docs_count())
print("Cranfield queries:", cranfield.queries_count())

print("Vaswani docs:", vaswani.docs_count())
print("Vaswani queries:", vaswani.queries_count())

Cranfield docs: 1400
Cranfield queries: 225
Vaswani docs: 11429
Vaswani queries: 93


In [5]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

nltk.download('stopwords')

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [6]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", "", text)
    tokens = text.split()
    tokens = [t for t in tokens if t not in stop_words]
    tokens = [stemmer.stem(t) for t in tokens]
    return tokens

In [7]:
doc = next(vaswani.docs_iter())
print("ORIGINAL:")
print(doc.text[:400])

print("\nPROCESSED:")
print(preprocess(doc.text[:400]))

ORIGINAL:
compact memories have flexible capacities  a digital data storage
system with capacity up to bits and random and or sequential access
is described


PROCESSED:
['compact', 'memori', 'flexibl', 'capac', 'digit', 'data', 'storag', 'system', 'capac', 'bit', 'random', 'sequenti', 'access', 'describ']


In [8]:
cran_docs = {doc.doc_id: preprocess(doc.text) for doc in cranfield.docs_iter()}
cran_queries = {q.query_id: preprocess(q.text) for q in cranfield.queries_iter()}
vas_docs = {doc.doc_id: preprocess(doc.text) for doc in vaswani.docs_iter()}
vas_queries = {q.query_id: preprocess(q.text) for q in vaswani.queries_iter()}

In [9]:
from collections import defaultdict, Counter

def build_inverted_index(dataset, preprocess_fn):
    
    inverted_index = defaultdict(dict)
    doc_lengths = {}

    for doc in dataset.docs_iter():

        
        text = getattr(doc, "text", "")

        
        tokens = preprocess_fn(text)

        
        doc_lengths[doc.doc_id] = len(tokens)

        
        term_counts = Counter(tokens)

        
        for term, freq in term_counts.items():
            inverted_index[term][doc.doc_id] = freq

    return dict(inverted_index), doc_lengths

In [10]:
cranfield_index, cranfield_doc_lengths = build_inverted_index(
    cranfield,
    preprocess
)

In [11]:
vaswani_index, vaswani_doc_lengths = build_inverted_index(
    vaswani,
    preprocess
)

In [ ]:
print("Unique terms:", len(vaswani_index))

sample_word = list(vaswani_index.keys())[7]

print("Sample term:", sample_word)

print(vaswani_index[sample_word])

Unique terms: 7898
Sample term: system
{'1': 1, '2': 1, '3': 1, '5': 2, '13': 1, '22': 1, '24': 2, '42': 1, '129': 1, '142': 1, '144': 2, '159': 1, '179': 1, '188': 1, '197': 1, '202': 3, '265': 1, '291': 1, '302': 2, '314': 1, '317': 1, '321': 1, '383': 1, '392': 1, '398': 1, '402': 2, '406': 1, '410': 1, '413': 2, '416': 1, '429': 1, '435': 1, '502': 2, '506': 1, '525': 1, '558': 1, '572': 1, '637': 1, '638': 1, '648': 1, '691': 1, '736': 2, '752': 1, '773': 1, '775': 1, '779': 1, '795': 1, '800': 1, '803': 1, '805': 1, '828': 1, '844': 1, '847': 2, '879': 1, '884': 1, '918': 1, '921': 2, '923': 1, '924': 1, '932': 1, '965': 1, '966': 1, '1010': 1, '1018': 1, '1033': 1, '1049': 1, '1050': 1, '1051': 2, '1076': 1, '1077': 1, '1097': 1, '1158': 1, '1159': 2, '1181': 1, '1189': 1, '1196': 1, '1201': 1, '1227': 1, '1244': 1, '1257': 1, '1258': 1, '1262': 1, '1264': 1, '1294': 1, '1297': 4, '1300': 1, '1303': 1, '1313': 1, '1328': 1, '1334': 3, '1345': 1, '1368': 1, '1374': 1, '1404': 2, 